<a href="https://colab.research.google.com/github/laisalegri/curso-fiap-agentes-autonomos-ia/blob/main/FIAP_hands_on_3_agentes_pydanticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pydantic
Pydantic é uma biblioteca do Python utilizada para validar, organizar e garantir que os dados estejam no formato correto.

Em termos simples, funciona como um fiscal dos dados: verifica se as informações recebidas atendem às regras definidas antes de serem utilizadas pelo sistema.

In [1]:
pip install pydantic-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 119.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.2/956.2 kB 59.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 122.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 126.7 MB/s  0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentele

In [ ]:
from pydantic import BaseModel, Field
from decimal import Decimal
from typing import Optional

class Produto(BaseModel):
    nome: str = Field(min_length=1, max_length=100)
    preco: Decimal = Field(gt=0, le=10000)
    descricao: Optional[str] = Field(None, max_length=500)
    categoria: str = Field( ... , pattern=r'^[A-Za-z\s]+$')
    em_estoque: int = Field(ge=0)
    disponivel: bool = True

In [ ]:
try:
    produto_invalido = Produto(nome="", preco =- 5, categoria="1234", em_estoque =- 10)
except Exception as e:
    print("Erros de validação:", e)

Erros de validação: 4 validation errors for Produto
nome
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
preco
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than
categoria
  String should match pattern '^[A-Za-z\s]+$' [type=string_pattern_mismatch, input_value='1234', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_pattern_mismatch
em_estoque
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal


# Pydanctic AI + LLM
Integrar com IA e LLM para evitar alucinação.

In [1]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (300 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directo

In [2]:
# 1. Instala Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Inicia servidor Ollama server em background
!nohup ollama serve > /dev/null 2>&1 &

# 3. Faz download do modelo
!ollama pull gpt-oss:20b

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
import nest_asyncio

# Apply the patch to allow nested asyncio event loops
nest_asyncio.apply()

class CityLocation(BaseModel):
    city: str
    country: str

ollama_model = OpenAIChatModel(
    model_name='gpt-oss:20b', # must match the model you pulled in Ollama
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)

agent = Agent(model=ollama_model, output_type=CityLocation)

result = agent.run_sync('Where were the Olympics held in 2012?')
print(result.output) # -> city='London' country='United Kingdom'

city='London' country='United Kingdom'


# Ferramentas, Toolsets e Injeção de Dependências

In [ ]:
import random
from pydantic_ai import Agent, RunContext

agent = Agent (
    model=ollama_model,
    deps_type=str,
    system_prompt="Voce e um jogo de dados: role o dado e veja se acerta o palpite."
)

@agent.tool_plain
def roll_dice() -> str:
    return str(random.randint(1, 6)) # função simula o lançamento de um dado

@agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    return ctx.deps

dice_result = agent.run_sync('Meu palpite e 4', deps='Laís')
print(dice_result.output)

Rolou um 2.  
Seu palpite era 4, então não acertou.  
Você pode tentar novamente!


# Criando Agentes de IA com PydanticAI

## Agente de Atendimento Médico

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Protocol
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

ollama_model = OpenAIChatModel(
    model_name='gpt-oss:20b', # must match the model you pulled in Ollama
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)


# --- Contrato para o seu banco (assincrono)
class MedicalDatabase(Protocol):
    async def get_history(self, patient_id: int) -> str: ...
    async def get_patient_name(self, patient_id: int) -> str: ...


# --- Dependências do agente (injeção)
@dataclass
class RequestContext:
    patient_id: int
    db: MedicalDatabase


# --- Esquemas de entrada/saída ---
class AssistanceRequest(BaseModel):
    symptoms: str = Field(max_length=500)
    urgency: int = Field(ge=1, le=5)

class AssistanceResponse(BaseModel):
    advice: str
    risk_level: int = Field(ge=0, le=10)
    refer_to_specialist: bool

# --- Definição do agente
assist_agent = Agent(
    model=ollama_model,
    deps_type=RequestContext,
    output_type=AssistanceResponse,
    # Use "instructions" em vez de "system_prompt" para o comportamento atual recomendado
    instructions=(
        "Voce e um assistente medico virtual. "
        "Avalie os sintomas de forma prudente, comunique incertezas, "
        "e forneca orientacao geral e nao-diagnostica. "
        "Se houver sinais de emergencia (ex .: dor toracica intensa, falta de ar importante), "
        "incentive procurar atendimento imediato (SAMU/192 ou emergencia). "
        "Respeite o limite do escopo: nao prescreva medicamentos."
    ),
)

# --- Tool que usa o contexto (RunContext[RequestContext])
@assist_agent.tool
async def fetch_patient_history(ctx: RunContext[RequestContext]) -> str:
    """Busca historico clinico do paciente no banco."""
    return await ctx.deps.db.get_history(patient_id=ctx.deps.patient_id)

# --- Prompt dinamico: acrescenta contexto do paciente antes da execução
@assist_agent.system_prompt
async def add_patient_context(ctx: RunContext[RequestContext]) -> str:
    name = await ctx.deps.db.get_patient_name(ctx. deps.patient_id)
    return f"Contexto: paciente '{name}' (id={ctx.deps.patient_id})."

# --- Exemplo de execução (sync)
class FakeDB:
    async def get_history(self, patient_id: int) -> str:
        return "Hipertensao controlada; sem alergias registradas."
    async def get_patient_name(self, patient_id: int) -> str:
        return "Jose da Silva"

deps = RequestContext(patient_id=42, db=FakeDB())
question = AssistanceRequest(symptoms="Dor no peito e falta de ar ao esforco", urgency=5)
result = await assist_agent.run(
    f"Avalie: {question.model_dump_json()}",
    deps=deps,
)
print(result.output)

advice='Seu relato de dor torácica acompanhada de falta de ar ao esforço é preocupante, principalmente considerando a presença de hipertensão. Esses sintomas podem indicar problemas cardíacos (como angina ou doença arterial coronariana) ou pulmonares (como doença pulmonar obstrutiva ou insuficiência cardíaca). Mesmo que a dor não seja intensa no momento, qualquer episódio de desconforto no peito, especialmente quando ocorre com esforço, deve ser avaliado de forma urgente. \n\n**O que fazer agora:**\n1. Se a dor torácica piorar, se espalhar para ombro, braço, pescoço ou mandíbula, ou se houver sudorese, náusea, tontura ou sensação de desmaio, procure atendimento de emergência imediatamente (ligue 192/SAMU ou vá ao pronto‑socorro mais próximo). \n2. Se a dor for leve a moderada e não piorar com repouso, agende uma consulta com um cardiologista na maior brevidade possível. Informe ao médico sobre a hipertensão e quaisquer fatores de risco adicionais (fumar, colesterol alto, histórico fami

# Múltiplos Agentes

In [5]:
from dataclasses import dataclass
from typing import Protocol
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai.usage import UsageLimits
import asyncio

ollama_model = OpenAIChatModel(
    model_name='gpt-oss:20b', # must match the model you pulled in Ollama
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)

joke_selection_agent = Agent(
    ollama_model,
    system_prompt='Use a `joke_factory` para gerar piadas, depois escolha a melhor e retorne apenas uma.'
)

joke_generation_agent = Agent(
    ollama_model,
    output_type=list[str]
)

@joke_selection_agent.tool
async def joke_factory(ctx: RunContext[None], count: int) -> list[str]:
    r = await joke_generation_agent.run(
        f'Por favor, gere {count} piadas.',
        usage=ctx.usage,
    )
    return r.output

async def main():
    result = await joke_selection_agent.run(
        'Conte uma piada.',
        usage_limits=UsageLimits(request_limit=5, total_tokens_limit=2000),
    )
    print(result.output)
    print(result.usage())

await main()

O que o zero disse ao oito? Que você tem um belo cinto!
RunUsage(cache_read_tokens=495, input_tokens=584, output_tokens=287, requests=3, tool_calls=1)


/tmp/ipykernel_6760/971700655.py:39: PydanticAIDeprecationWarning: `usage` is a property, not a method: use `.usage` instead of `.usage()`.
  print(result.usage())
